# Exploratory Data Analysis

The task is ranking candidate profiles against a Senior AI Engineer job description.
This is not a pure similarity problem: the dataset is adversarial by design, profiles
that match on keywords but not on substance are planted to penalize naive approaches.

Objective of this notebook:
- Map the schema to the fields that carry real signal
- Quantify how often declared skills disagree with actual roles
- Profile the behavioral signals that gate candidate availability



In [2]:
!pip install pandas numpy matplotlib -q


## Data loading

The pool ships as gzipped JSONL. A `LIMIT` is used during iteration; set it to `None`
for the full pool. The loader also accepts a plain JSON sample.


In [6]:
import gzip, json
import pandas as pd

PATH = r"C:\Users\chenk\automated-recuitment-system\data\raw\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\candidates.jsonl"
LIMIT = 5000   # None for the full 100K pool

def load(path, limit=None):
    rows = []
    opener = gzip.open if path.endswith(".gz") else open
    if path.endswith(".json"):
        with open(path) as f:
            return json.load(f)
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rows.append(json.loads(line))
            if limit and len(rows) >= limit:
                break
    return rows

cands = load(PATH, LIMIT)
print(f"Loaded {len(cands):,} candidates")


Loaded 5,000 candidates


## Feature flattening

Profiles are nested. The relevant signal concentrates in a few fields: current and
historical titles, the free-text role descriptions, declared skills, and behavioral
metrics. Title/history and skills/summary are kept separate, the hypothesis is that the
former are reliable and the latter are susceptible to keyword inflation.


In [ ]:
def flatten(c):
    p = c["profile"]
    sig = c["redrob_signals"]
    history_text = " ".join(h["title"] + ": " + h["description"]
    for h in c["career_history"])
    titles = [h["title"] for h in c["career_history"]]
    skills = [s["name"] for s in c.get("skills", [])]
    return {
        "candidate_id": c["candidate_id"],
        "current_title": p["current_title"],
        "yoe": p["years_of_experience"],
        "headline": p["headline"],
        "summary": p["summary"],
        "career_titles": " | ".join(titles),
        "history_text": history_text,
        "skills": ", ".join(skills),
        "n_skills": len(skills),
        "github": sig["github_activity_score"],
        "last_active": sig["last_active_date"],
        "response_rate": sig["recruiter_response_rate"],
        "open_to_work": sig["open_to_work_flag"],
    }

df = pd.DataFrame(flatten(c) for c in cands)
print(df.shape)
df.head()

(5000, 13)


,candidate_id,current_title,yoe,headline,summary,career_titles,history_text,skills,n_skills,github,last_active,response_rate,open_to_work
0,CAND_0000001,Backend Engineer,6.9,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,Backend Engineer | Analytics Engineer,Backend Engineer: Implemented streaming data p...,"Tailwind, NLP, Image Classification, Fine-tuni...",17,9.2,2026-05-20,0.34,True
1,CAND_0000002,Operations Manager,12.5,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,Operations Manager | Operations Manager | Mark...,Operations Manager: Customer support team lead...,"Project Management, React, Photoshop, TypeScri...",9,-1.0,2025-11-12,0.29,True
2,CAND_0000003,Customer Support,1.1,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,Customer Support,Customer Support: Business analyst at a consul...,"Angular, SEO, Excel, Accounting, Kubernetes, D...",6,-1.0,2026-03-21,0.46,False
3,CAND_0000004,Marketing Manager,3.8,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,Marketing Manager | Operations Manager | Busin...,Marketing Manager: Mechanical engineering desi...,"Node.js, Content Writing, Redux, Airflow, Grap...",10,-1.0,2026-03-25,0.26,False
4,CAND_0000005,Accountant,11.0,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,Accountant | HR Manager | HR Manager | Accountant,Accountant: Business analyst at a consulting f...,"SQL, PowerPoint, Photoshop, Tailwind, Apache F...",6,-1.0,2025-10-01,0.37,True


## Skill–title consistency

A reliable dataset would show AI-heavy skill sets concentrated in technical roles. The
check below counts profiles with multiple AI skills but a non-technical current title.
A high count confirms the keyword-inflation pattern and means skill-based features must
be down-weighted relative to role evidence.


In [8]:
AI_SKILLS = ["embeddings", "retrieval", "ranking", "nlp", "llm", "machine learning","faiss", "pinecone", "sentence transformers", "transformers", "mlops"]
TECH_TITLES = ["engineer", "ml", "machine learning", "data scientist", "developer","scientist", "ai", "software"]

df["ai_skills"] = df["skills"].str.lower().apply(lambda s: sum(k in s for k in AI_SKILLS))
df["tech_title"] = df["current_title"].str.lower().apply(lambda t: any(k in t for k in TECH_TITLES))

mismatch = df[(df["ai_skills"] >= 2) & (~df["tech_title"])]
print(f"AI-heavy skills with non-technical title: {len(mismatch)} / {len(df)} "
      f"({len(mismatch)/len(df):.1%})")
mismatch[["candidate_id", "current_title", "ai_skills", "skills"]].head(10)


AI-heavy skills with non-technical title: 241 / 5000 (4.8%)


,candidate_id,current_title,ai_skills,skills
20,CAND_0000021,Project Manager,4,"Hadoop, PostgreSQL, Kafka, Microservices, AWS,..."
73,CAND_0000074,Operations Manager,7,"Go, gRPC, Next.js, Webpack, AWS, Information R..."
81,CAND_0000082,Data Analyst,4,"GANs, REST APIs, TTS, HTML, YOLO, SAP, Pinecon..."
82,CAND_0000083,Graphic Designer,5,"BigQuery, Flask, Figma, SAP, Vue.js, Marketing..."
119,CAND_0000120,Graphic Designer,5,"FastAPI, AWS, Redux, Marketing, SQL, React, Sa..."
120,CAND_0000121,Customer Support,6,"Airflow, Marketing, Excel, SEO, Sales, FAISS, ..."
132,CAND_0000133,Graphic Designer,4,"Airflow, Snowflake, Node.js, Hadoop, Accountin..."
200,CAND_0000201,Marketing Manager,4,"Sales, Data Pipelines, Vue.js, GraphQL, Market..."
202,CAND_0000203,Operations Manager,5,"Project Management, Terraform, Spark, HTML, No..."
210,CAND_0000211,Operations Manager,5,"Content Writing, dbt, Angular, Object Detectio..."


## Behavioral signal distribution

Fit is necessary but not sufficient: an unreachable candidate has no hiring value. These
distributions (experience, GitHub activity, recruiter response rate, recency of activity)
establish baselines before the signals are used as a multiplier on the fit score.


In [9]:
df["last_active"] = pd.to_datetime(df["last_active"])
print(df[["yoe", "github", "response_rate"]].describe())
print("\nGitHub linked (score > -1):", (df["github"] > -1).mean().round(2))
print("Open to work:", df["open_to_work"].mean().round(2))
df["last_active"].dt.to_period("M").value_counts().sort_index().tail(8)


               yoe       github  response_rate
count  5000.000000  5000.000000    5000.000000
mean      7.110900     9.505220       0.436274
std       3.800649    17.829217       0.214272
min       1.000000    -1.000000       0.040000
25%       3.900000    -1.000000       0.250000
50%       6.800000    -1.000000       0.430000
75%       9.900000    16.200000       0.620000
max      15.000000    96.900000       0.910000

GitHub linked (score > -1): 0.34
Open to work: 0.37


last_active
2025-10    406
2025-11    440
2025-12    654
2026-01    698
2026-02    649
2026-03    749
2026-04    756
2026-05    626
Freq: M, Name: count, dtype: int64

## Takeaways

1. **Role evidence outranks declared skills.** The skill–title mismatch rate confirms
   keyword inflation; current title and career-history text are the dependable signals.
2. **Availability is a separate axis.** Behavioral signals will modify the fit score so
   that strong-but-inactive profiles fall below strong-and-engaged ones.